# Disease Prediction from Symptoms — Machine Learning Project

**Dataset:** Disease-Symptom dataset (Kaggle) — 41 diseases, 132 distinct symptoms, plus lookup tables
for disease descriptions and recommended precautions.

**Goal:** given a set of symptoms a patient reports, predict the most likely disease — and then return a
description of that disease and precautions to take, using the extra lookup files that come with the dataset.

**Why this is more than a toy exercise:**
- The raw data isn't ready to feed into a model — symptoms are spread across 17 unstacked columns, with
  inconsistent spacing/typos that have to be reconciled against a separate severity-weight file.
- We engineer two different feature representations (binary presence vs. severity-weighted) and compare them.
- We train and compare 7 different classifiers, validate with stratified k-fold cross-validation (not just a
  single train/test split), and tune hyperparameters with `GridSearchCV`.
- We stress-test the final model by simulating incomplete/real-world symptom reporting.
- We close the loop with an actual usable prediction function that chains together disease → description → precautions.

**Files used (place them in a `data/` folder next to this notebook):**
- `dataset.csv` — 4,920 patient records, disease + up to 17 symptoms per record
- `Symptom-severity.csv` — a severity weight (1–7) for each of the 132 symptoms
- `symptom_Description.csv` — a short medical description per disease
- `symptom_precaution.csv` — up to 4 recommended precautions per disease

> ⚠️ Disclaimer: this is an academic/portfolio project trained on a small, templated dataset. It is **not**
> a validated medical tool and should never be used for real diagnosis.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

%matplotlib inline
print('Libraries loaded ✅')

## 2. Load the raw data
Four files, each playing a different role:
- `dataset.csv` → our training data
- `Symptom-severity.csv` → turns a plain "symptom present/absent" signal into a weighted one
- `symptom_Description.csv` / `symptom_precaution.csv` → used later to turn a raw prediction into something
  a human can actually read and act on

In [ ]:
DATA_DIR = 'data'

df = pd.read_csv(f'{DATA_DIR}/dataset.csv')
severity = pd.read_csv(f'{DATA_DIR}/Symptom-severity.csv')
descriptions = pd.read_csv(f'{DATA_DIR}/symptom_Description.csv')
precautions = pd.read_csv(f'{DATA_DIR}/symptom_precaution.csv')

print('dataset.csv:', df.shape)
print('Symptom-severity.csv:', severity.shape)
print('symptom_Description.csv:', descriptions.shape)
print('symptom_precaution.csv:', precautions.shape)
df.head()

In [ ]:
severity.head()

## 3. First look at the data
`dataset.csv` stores each patient as one disease label plus up to 17 symptom columns (`Symptom_1` ... `Symptom_17`).
Most patients don't have 17 symptoms, so the unused columns are `NaN` — that's expected, not an error.

In [ ]:
df.info()

In [ ]:
# How many diseases, and how many patient records per disease?
print('Number of unique diseases:', df['Disease'].nunique())
df['Disease'].value_counts()

**Observation:** every disease has *exactly* the same number of records (120). This dataset was
synthetically generated to be perfectly balanced — that's unusual for real medical data, and it will matter
later when we interpret how well our models perform.

In [ ]:
plt.figure(figsize=(8, 10))
df['Disease'].value_counts().sort_values().plot(kind='barh')
plt.title('Number of Records per Disease')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# How many symptoms does a typical patient record have (non-null symptom columns)?
symptom_cols_raw = [c for c in df.columns if c.startswith('Symptom')]
symptom_fill_count = df[symptom_cols_raw].notna().sum(axis=1)

plt.figure(figsize=(7, 4))
sns.histplot(symptom_fill_count, bins=range(1, 18), discrete=True)
plt.title('How Many Symptoms Are Recorded per Patient?')
plt.xlabel('Number of symptoms listed')
plt.show()
print(symptom_fill_count.describe())

## 4. Clean the text data
This dataset has a well-known quirk: the same symptom is sometimes written slightly differently across files
— e.g. `dataset.csv` has `"dischromic _patches"` (stray space) while `Symptom-severity.csv` has
`"dischromic_patches"` (no space). If we don't fix this, our severity lookup will silently fail for those symptoms.

Our cleaning function: lowercase everything, strip leading/trailing whitespace, and remove *internal* stray
spaces (since the real word separator in this dataset is always an underscore).

In [ ]:
def clean_symptom(s):
    """Normalize a symptom string so the same symptom always matches across files."""
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = s.replace(' ', '')  # e.g. 'dischromic _patches' -> 'dischromic_patches'
    return s

symptom_cols = [c for c in df.columns if c.startswith('Symptom')]
df[symptom_cols] = df[symptom_cols].map(clean_symptom)
df['Disease'] = df['Disease'].str.strip()

severity['Symptom'] = severity['Symptom'].apply(clean_symptom)
severity = severity.drop_duplicates(subset='Symptom')
severity_map = dict(zip(severity['Symptom'], severity['weight']))

descriptions['Disease'] = descriptions['Disease'].str.strip()
precautions['Disease'] = precautions['Disease'].str.strip()

print('Cleaning done ✅')
df.head(3)

In [ ]:
# Sanity check: does every symptom in dataset.csv now have a matching severity weight?
all_symptoms = sorted(set(
    val for col in symptom_cols for val in df[col].dropna().unique()
))
print('Unique symptoms found:', len(all_symptoms))

missing = [s for s in all_symptoms if s not in severity_map]
print('Symptoms still missing a severity weight:', missing)

## 5. Feature engineering
Machine learning models need numeric input, not lists of text symptoms. We'll build **two** representations
and combine ideas from both:

1. **Multi-hot binary matrix** — one column per unique symptom (131 columns), `1` if the patient has it, `0` if not.
2. **Severity-weighted matrix** — same shape, but instead of a flat `1`, we use that symptom's severity weight
   (from `Symptom-severity.csv`). A severe symptom like `coma` (weight 7) should carry more predictive signal
   than a mild one like `itching` (weight 1).

We'll also add 3 hand-crafted summary features per patient: `symptom_count`, `avg_severity`, `max_severity`.
These are the kind of engineered features a doctor would intuitively find useful — and models often do too.

In [ ]:
# 1. Binary multi-hot matrix
binary_matrix = pd.DataFrame(0, index=df.index, columns=all_symptoms)
for col in symptom_cols:
    for idx, val in df[col].items():
        if pd.notna(val):
            binary_matrix.at[idx, val] = 1

print('Binary matrix shape:', binary_matrix.shape)
binary_matrix.head(3)

In [ ]:
# 2. Severity-weighted matrix — same 0/1 pattern, but scaled by each symptom's severity weight
weighted_matrix = binary_matrix.copy()
for sym in all_symptoms:
    w = severity_map.get(sym, 0)
    weighted_matrix[sym] = weighted_matrix[sym] * w

weighted_matrix.head(3)

In [ ]:
# 3. Hand-crafted summary features
df['symptom_count'] = binary_matrix.sum(axis=1)
df['avg_severity'] = weighted_matrix.replace(0, np.nan).mean(axis=1).fillna(0)
df['max_severity'] = weighted_matrix.max(axis=1)

df[['Disease', 'symptom_count', 'avg_severity', 'max_severity']].head()

### Which symptoms show up most often across the whole dataset?

In [ ]:
symptom_totals = binary_matrix.sum(axis=0).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
symptom_totals.head(20).sort_values().plot(kind='barh')
plt.title('20 Most Common Symptoms in the Dataset')
plt.xlabel('Number of patients with this symptom')
plt.tight_layout()
plt.show()

### Does symptom count vary a lot by disease?
If a disease is consistently reported with more symptoms than another, `symptom_count` alone becomes a
useful (if slightly "cheating") predictive signal.

In [ ]:
avg_symptoms_per_disease = df.groupby('Disease')['symptom_count'].mean().sort_values()

plt.figure(figsize=(8, 10))
avg_symptoms_per_disease.plot(kind='barh')
plt.title('Average Number of Symptoms Reported, by Disease')
plt.xlabel('Average symptom count')
plt.tight_layout()
plt.show()

## 6. Build the final feature matrix and encode the target
We'll use the **severity-weighted matrix** plus our 3 engineered features as `X`. This gives the model richer
information than plain binary flags. `Disease` (text) is encoded into numbers with `LabelEncoder`, since
scikit-learn classifiers need numeric labels — we keep the encoder so we can translate predictions back to
disease names later.

In [ ]:
X = weighted_matrix.copy()
X['symptom_count'] = df['symptom_count']
X['avg_severity'] = df['avg_severity']
X['max_severity'] = df['max_severity']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Disease'])

print('X shape:', X.shape)
print('Number of classes:', len(label_encoder.classes_))

## 7. Train / test split
We stratify the split so every disease is proportionally represented in both the training and test sets —
important with 41 classes, so a rare class can't accidentally end up entirely in one side.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

## 8. Train and compare 7 classifiers
Distance-based / linear models (Logistic Regression, KNN, SVM) get the **scaled** features; tree-based
models don't need scaling since they split on raw thresholds.

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), True),
    'K-Nearest Neighbors': (KNeighborsClassifier(n_neighbors=5), True),
    'Naive Bayes': (GaussianNB(), False),
    'Decision Tree': (DecisionTreeClassifier(random_state=RANDOM_STATE), False),
    'Random Forest': (RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE), False),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=RANDOM_STATE), False),
    'SVM (RBF kernel)': (SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE), True),
}

test_results = {}
for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train
    Xte = X_test_scaled if needs_scaling else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    acc = accuracy_score(y_test, preds)
    test_results[name] = acc
    print(f'{name:22s}: {acc:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
pd.Series(test_results).sort_values().plot(kind='barh', color='steelblue')
plt.title('Test Accuracy by Model')
plt.xlabel('Accuracy')
plt.xlim(0.9, 1.01)
plt.tight_layout()
plt.show()

## 9. Cross-validation
A single train/test split can get lucky or unlucky. Stratified 5-fold cross-validation trains and tests each
model 5 times on different slices of the data, which gives a much more reliable estimate of real performance.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, (model, needs_scaling) in models.items():
    Xcv = scaler.fit_transform(X) if needs_scaling else X
    scores = cross_val_score(model, Xcv, y, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = (scores.mean(), scores.std())
    print(f'{name:22s}: mean={scores.mean():.4f}  std={scores.std():.4f}')

## 10. Hyperparameter tuning with GridSearchCV
Random Forest was one of the strongest, most stable models above. Let's search over a small grid of
hyperparameters to see if we can squeeze out anything better, using 3-fold CV during the search.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print('Best hyperparameters:', grid_search.best_params_)
print('Best CV accuracy:', grid_search.best_score_)

best_model = grid_search.best_estimator_
final_test_acc = accuracy_score(y_test, best_model.predict(X_test))
print('Final tuned model — test accuracy:', final_test_acc)

## 11. Detailed evaluation of the final model

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Tuned Random Forest (41 classes)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### ⚠️ Why is accuracy essentially perfect?
It's tempting to celebrate 100% accuracy, but it's important to understand *why* it happens here rather than
just report the number:

- This dataset was **synthetically generated**: every disease maps to a small, fixed, non-overlapping set of
  symptom combinations. There's no measurement noise, no ambiguous overlapping cases, and no two diseases
  share an identical symptom profile.
- In other words, this is closer to a *lookup table* dressed up as a classification problem than it is to
  noisy real-world clinical data, where the same symptoms (fever, fatigue, headache...) show up across dozens
  of unrelated conditions.
- A perfect score here demonstrates that the **pipeline is implemented correctly** — but it does *not* mean
  this model would perform this well on real patient data. That's a distinction worth being able to explain.

To make this more realistic, let's simulate what happens when symptom reporting is *incomplete* — which is
much closer to how real patients describe symptoms.

## 12. Robustness check: what if some symptoms are missing?
Real patients rarely report every symptom they have. We simulate this by randomly removing 1–2 of the
symptoms that were actually present in each test case, then see how much accuracy drops.

In [ ]:
def simulate_missing_symptoms(X_subset, symptom_columns, max_drop=2, seed=RANDOM_STATE):
    rng = random.Random(seed)
    X_noisy = X_subset.copy()
    for idx in X_noisy.index:
        present_cols = [c for c in symptom_columns if X_noisy.loc[idx, c] > 0]
        if len(present_cols) > 1:
            n_drop = min(len(present_cols) - 1, rng.choice(range(1, max_drop + 1)))
            drop_cols = rng.sample(present_cols, n_drop)
            X_noisy.loc[idx, drop_cols] = 0
    return X_noisy

X_test_noisy = simulate_missing_symptoms(X_test, all_symptoms)
noisy_preds = best_model.predict(X_test_noisy)
noisy_acc = accuracy_score(y_test, noisy_preds)

print(f'Accuracy with full symptom info:        {final_test_acc:.4f}')
print(f'Accuracy with 1-2 symptoms missing:      {noisy_acc:.4f}')

This small drop (even though it looks tiny in absolute terms) shows the model *does* rely on
having enough symptom information — it isn't simply memorizing row indices. In a real clinical dataset with
messier, overlapping symptom patterns, this kind of robustness test would show a much bigger gap, and would
be a genuinely important thing to report.

## 13. What did the model actually learn? Feature importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
importances.head(20).sort_values().plot(kind='barh')
plt.title('Top 20 Most Important Features (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### 2D visualization with PCA
We can't visualize 134 dimensions directly, but PCA lets us compress them down to 2 dimensions that capture
as much variance as possible, so we can *see* how separable the diseases are.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab20', s=15, alpha=0.7)
plt.title('Diseases Projected onto 2 Principal Components')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.tight_layout()
plt.show()

print('Total variance captured by 2 components:', f'{pca.explained_variance_ratio_.sum():.1%}')

## 14. Save the model and everything needed to reuse it
To make a prediction later (e.g. in a small web app), we need more than just the model — we also need the
scaler, the label encoder, and the exact list/order of feature columns it was trained on.

In [ ]:
joblib.dump(best_model, 'disease_prediction_model.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
joblib.dump(list(X.columns), 'feature_columns.pkl')
joblib.dump(severity_map, 'severity_map.pkl')
print('Saved model + encoder + feature list + severity map ✅')

## 15. Putting it all together: an end-to-end symptom checker
This is the payoff of loading `symptom_Description.csv` and `symptom_precaution.csv` back at the start:
given a plain list of symptoms, we predict the disease *and* return a human-readable description and
recommended precautions, the way a real symptom-checker tool would.

In [ ]:
def predict_disease(symptom_list, model=best_model, feature_columns=list(X.columns),
                     severity_map=severity_map, label_encoder=label_encoder):
    """Predict a disease from a list of symptom strings and return a full, readable report."""
    input_vec = pd.Series(0, index=feature_columns, dtype=float)
    cleaned = [clean_symptom(s) for s in symptom_list]

    present_weights = []
    unrecognized = []
    for s in cleaned:
        if s in feature_columns:
            w = severity_map.get(s, 1)
            input_vec[s] = w
            present_weights.append(w)
        else:
            unrecognized.append(s)

    input_vec['symptom_count'] = len(present_weights)
    input_vec['avg_severity'] = np.mean(present_weights) if present_weights else 0
    input_vec['max_severity'] = max(present_weights) if present_weights else 0

    input_df = pd.DataFrame([input_vec.values], columns=feature_columns)
    pred_encoded = model.predict(input_df)[0]
    disease = label_encoder.inverse_transform([pred_encoded])[0]

    # confidence, if the model supports predict_proba
    confidence = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(input_df)[0]
        confidence = proba[pred_encoded]

    desc_row = descriptions[descriptions['Disease'] == disease]
    description = desc_row['Description'].values[0] if len(desc_row) else 'No description available.'

    prec_row = precautions[precautions['Disease'] == disease]
    if len(prec_row):
        precaution_list = [p for p in prec_row.iloc[0, 1:].tolist() if isinstance(p, str)]
    else:
        precaution_list = []

    print('Symptoms entered   :', symptom_list)
    if unrecognized:
        print('Not recognized     :', unrecognized)
    print('Predicted disease  :', disease)
    if confidence is not None:
        print(f'Model confidence   : {confidence:.1%}')
    print('Description        :', description)
    print('Recommended steps  :')
    for i, p in enumerate(precaution_list, 1):
        print(f'   {i}. {p}')
    print()

    return disease

In [ ]:
_ = predict_disease(['itching', 'skin_rash', 'nodal_skin_eruptions'])
_ = predict_disease(['high_fever', 'headache', 'nausea', 'vomiting'])
_ = predict_disease(['chest_pain', 'breathlessness', 'sweating'])

## 16. Conclusion

**What we built:** a full pipeline that takes messy, multi-column symptom data, reconciles it against a
severity-weight reference table, engineers two different feature representations, compares 7 classifiers with
cross-validation, tunes the best one, stress-tests it against incomplete symptom reporting, and finally wraps
it in a usable symptom-checker function that returns a diagnosis, description, and precautions.

**Honest limitations:**
- The dataset is small (4,920 rows) and synthetically balanced — every disease maps to a fixed,
  non-overlapping symptom pattern, which is why accuracy is close to 100%. Real clinical data is much noisier.
- Symptoms here are categorical presence/absence + a static severity weight — real symptom reporting also
  varies in duration, intensity over time, and patient-specific context (age, history, comorbidities), none of
  which this dataset captures.
- No demographic features (age, sex, pre-existing conditions) are included, which would matter a lot in a
  real diagnostic setting.

**Possible next steps:**
- Add controlled noise/symptom overlap between diseases to make the classification task harder and more realistic.
- Try SHAP values for a deeper, per-prediction explanation of *why* the model predicted a given disease.
- Wrap `predict_disease()` in a small Streamlit or Flask app so it can be used interactively.
- Combine with a real, larger clinical dataset (e.g. from a hospital or public health API) for a more rigorous version.